# ROGII - Wellbore Geology Prediction

**Reference:** 
- [rogii-sel15-rerun](https://www.kaggle.com/code/aidensong123/rogii-sel15-rerun)
- [[ROGII] BETTER SOLUTION | LB: 9.956](https://www.kaggle.com/code/romantamrazov/rogii-better-solution-lb-9-956)
- [[ROGII] SUPER SOLUTION |LB: TOP 3](https://www.kaggle.com/code/romantamrazov/rogii-super-solution-lb-top-3)
- [Top 2 Rank | 10.784 | Physics-Informed Baseline](https://www.kaggle.com/code/karnakbaevarthur/top-2-rank-10-784-physics-informed-baseline)
- [Triple-Signal Beam Search + Dual PF + LightGBM](https://www.kaggle.com/code/shinyanagai123/triple-signal-beam-search-dual-pf-lightgbm)
- [rogii plane fit formation top knn](https://www.kaggle.com/code/konbu17/rogii-plane-fit-formation-top-knn)
- [ROGII-Wellbore-Geology-Prediction](https://www.kaggle.com/code/vishwasmishra1234/rogii-wellbore-geology-prediction)
- [XGB Starter - [CV 15]](https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15)

**Original Notebook:**
- https://www.kaggle.com/code/ravaghi/wellbore-geology-prediction-ridge

## Enhanced: Kalman Filter + Particle Filter (RBPF / IMM Hybrid)

**Changes vs original notebook:**
- Added `run_kf_tvt()`: Extended Kalman Filter for smooth/linear wells
- Added `_pf_rbpf()`: Rao-Blackwellised PF — KF handles linear velocity, PF handles non-linear GR likelihood
- Added `run_imm_kf_pf()`: Interacting Multiple Model — switches between KF and RBPF based on GR uncertainty
- `build_well()` extended with 6 new KF/RBPF/IMM feature columns
- `run_pf_lik_ensemble_scales()` now also returns an IMM ensemble
- Selector updated to include `kf` and `imm` blend variants
- Final blend updated: sub_1 (ML) 0.25, sub_2 (Physics/IMM) 0.75

# 1. Imports and configs

In [ ]:
!pip install --no-index --find-links /kaggle/input/notebooks/kushubhai/rogii-wheel koolbox hill-climbing lightgbm catboost xgboost scikit-learn==1.7.2

In [ ]:
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from datetime import datetime, timedelta
from catboost import CatBoostRegressor
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from koolbox import Trainer
from pathlib import Path
from numba import njit
import multiprocessing
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
import joblib
import time
import glob
import os

warnings.filterwarnings("ignore")

In [ ]:
class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    models_path = Path("/kaggle/input/datasets/kushubhai/rogii-models")
    
    seed = 42
    n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    
    metric = root_mean_squared_error

# 2. Data loading and preprocessing

In [ ]:
SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)

SELECTOR_BIN_VARIANTS = {
    0: 'imm_scale_5_hold_0.2',       # was pf_scale_5_hold_0.2  → IMM better for smooth wells
    1: 'kf_hold_0.15',               # was pf_scale_3_hold_0.15 → pure KF for small n_eval smooth
    2: 'pf_scale_12_beam_0.2_hold_0.15',
    3: 'pf_scale_5_hold_0.15',
    4: 'pf_scale_5_beam_0.05_hold_0.05',
    5: 'pf_scale_12_beam_0.2_hold_0.05',
}

SELECTOR_GLOBAL_VARIANT = 'imm_scale_8_hold_0.2'
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

FORMATION_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]

def impute_gr_from_nearest(df,
                           gr_col="GR",
                           x_col="X",
                           y_col="Y",
                           z_col="Z"):
    """
    Imputes NaN GR values using the nearest
    available coordinate point.

    Returns:
        DataFrame with imputed GR values
    """

    df = df.copy()
    available_df = df[df[gr_col].notna()].copy()
    available_coords = available_df[[x_col, y_col, z_col]].values

    for idx in df[df[gr_col].isna()].index:
        missing_coord = df.loc[idx, [x_col, y_col, z_col]].values.astype(float)
        diffs = available_coords - missing_coord

        distances = np.sqrt(
            diffs[:, 0]**2 +
            diffs[:, 1]**2 +
            diffs[:, 2]**2
        )
        nearest_pos = np.argmin(distances)
        nearest_gr = available_df.iloc[nearest_pos][gr_col]

        df.at[idx, gr_col] = nearest_gr
    return df

def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def load_well(wid, split='train'):
    base = CFG.dataset_path / split
    hw = pd.read_csv(base / f'{wid}__horizontal_well.csv')
    tw = pd.read_csv(base / f'{wid}__typewell.csv')
    return hw, tw


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = impute_gr_from_nearest(tw_s)['GR'].values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks   = np.array(liks)
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, 0)).sum(0)


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    """Multi-seed multi-scale PF ensemble. Also returns IMM blend."""
    preds, liks = [], []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p); liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks     = np.array(liks)
    liks_n   = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
    out['pf_mean'] = pred_arr.mean(0)

    # ── NEW: also run KF and IMM, store alongside PF scales ──────────────────
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    try:
        kf_full = run_kf_tvt(hw, tw_tvt, tw_gr)
        out['kf'] = kf_full
    except Exception:
        out['kf'] = out['pf_scale_8']
    try:
        imm_full = run_imm_kf_pf(hw, tw_tvt, tw_gr)
        out['imm_scale_8'] = imm_full
        for scale in scales:
            out[f'imm_scale_{scale:g}'] = imm_full   # same IMM, alias per scale key
    except Exception:
        for scale in scales:
            out[f'imm_scale_{scale:g}'] = out[f'pf_scale_{scale:g}']

    return out


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


def selector_well_code(hw):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    code = n_bin + 2 * z_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span


def parse_selector_variant(name):
    """
    Extended parser that handles:
      pf_scale_5_hold_0.2
      pf_scale_12_beam_0.2_hold_0.15
      kf_hold_0.15
      imm_scale_8_hold_0.2
      imm_scale_5_beam_0.1_hold_0.1
    Returns (mode, scale, beam_weight, hold_weight)
    mode: 'pf' | 'kf' | 'imm'
    """
    parts = name.split('_')
    mode  = parts[0]                    # 'pf', 'kf', or 'imm'
    scale = 8.0
    if 'scale' in parts:
        scale = float(parts[parts.index('scale') + 1])
    beam_weight = 0.0
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    hold_weight = 0.0
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    return mode, scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    """
    Extended: supports kf and imm keys inside pf_by_scale dict.
    """
    mode, scale, beam_weight, hold_weight = parse_selector_variant(name)

    if mode == 'kf':
        base = pf_by_scale.get('kf', pf_by_scale.get('pf_scale_8'))
    elif mode == 'imm':
        base = pf_by_scale.get(f'imm_scale_{scale:g}',
               pf_by_scale.get('imm_scale_8',
               pf_by_scale.get('pf_scale_8')))
    else:  # 'pf'
        base = pf_by_scale.get(f'pf_scale_{scale:g}',
               pf_by_scale.get('pf_scale_8'))

    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    return pred

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# NEW: Extended Kalman Filter (EKF) for TVT tracking
# ═══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# FIX 2 — run_kf_tvt  (Q matrix only; everything else is unchanged)
# ══════════════════════════════════════════════════════════════════════════════
 
def run_kf_tvt(hw, tw_tvt, tw_gr,
               q_pos=0.08, q_vel=0.001,
               r_gr=None, r_vel=None):
    """
    Extended Kalman Filter for TVT estimation.
 
    State vector: x = [tvt, vel]   where vel = dTVT/dMD
    Observation:  z = [GR_observed, vel_from_inclination (optional)]
 
    Dynamics (constant-velocity model, linear):
        tvt_{k+1} = tvt_k + vel_k * dm
        vel_{k+1} = vel_k
 
    GR observation (non-linear → linearised via numerical Jacobian):
        h(tvt) = interp(tw_gr, tvt)   → gradient estimated numerically
 
    Parameters
    ----------
    q_pos : spectral density of position noise (m² / m)
    q_vel : spectral density of velocity (acceleration) noise (m² / m³)
    r_gr  : GR observation noise variance (auto-estimated if None)
    r_vel : velocity observation noise variance (auto-estimated if None)
 
    Returns
    -------
    Full TVT array (known section kept as-is, eval section filled by KF)
 
    CHANGE vs original — Q matrix
    ------------------------------
    The original code used:
        Q = np.diag([q_pos, q_vel]) * dm          # BUG: no off-diagonal terms
 
    This ignores the kinematic coupling: velocity noise integrated over dm
    accumulates into position noise with a covariance q_vel * dm² / 2.
    The correct continuous-white-noise-acceleration (CWNA) discretisation is:
 
        Q = | q_vel * dm³/3    q_vel * dm²/2 |   +  | q_pos * dm   0 |
            | q_vel * dm²/2    q_vel * dm    |      | 0            0 |
 
    The first term is the standard van Loan / Singer CWNA result; the second
    adds the independent position measurement noise that was already present.
    """
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()
 
    # ── Fit linear dynamics from known section ────────────────────────────────
    ktvt = kn['TVT_input'].values.astype(float)
    kmd  = kn['MD'].values.astype(float)
    kz   = kn['Z'].values.astype(float)
 
    # Estimate velocity prior from last 20 known points
    tail20  = kn.tail(20)
    dtvt_t  = np.diff(tail20['TVT_input'].values)
    dmd_t   = np.diff(tail20['MD'].values)
    mask_t  = dmd_t > 0
    vel0    = float(np.median(dtvt_t[mask_t] / dmd_t[mask_t])) if mask_t.sum() >= 3 else 0.0
 
    # Estimate beta: vel ~ beta * dz/dm + intercept  (inclination constraint)
    dz_k  = np.diff(kz)
    dvt_k = np.diff(ktvt)
    dmd_k = np.diff(kmd)
    m2    = dmd_k > 0
    if m2.sum() >= 10:
        vz_k  = dz_k[m2] / dmd_k[m2]
        vt_k  = dvt_k[m2] / dmd_k[m2]
        A_inc = np.column_stack([vz_k, np.ones_like(vz_k)])
        c_inc, _, _, _ = np.linalg.lstsq(A_inc, vt_k, rcond=None)
        beta_kf  = float(c_inc[0])
        icpt_kf  = float(c_inc[1])
        zsig_kf  = max(float(np.std(vt_k - (c_inc[0] * vz_k + c_inc[1]))), 0.005)
    else:
        beta_kf = -1.0
        icpt_kf =  0.0
        zsig_kf =  0.05
 
    # Auto-estimate observation noise from known section residuals
    gr_full  = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(
                    float(np.nanmean(tw_gr)))
    tw_at_kn = np.interp(ktvt, tw_tvt, tw_gr)
    gr_resid = kn['GR'].fillna(0).values - tw_at_kn
    r_gr_est  = float(np.clip(np.nanstd(gr_resid), 10., 60.))**2 if r_gr  is None else r_gr
    r_vel_est = zsig_kf**2                                         if r_vel is None else r_vel
 
    # ── Initialise state and covariance ───────────────────────────────────────
    x = np.array([float(ktvt[-1]), vel0])          # [tvt, vel]
    P = np.diag([4.0, (vel0 * 0.1 + 0.01)**2])    # initial covariance
 
    # NOTE: Q_base is removed; Q is now constructed analytically inside the loop
    # (see FIX comment in the loop body below).
 
    # Observation noise matrices (unchanged)
    R_gr  = np.array([[r_gr_est]])
    R_vel = np.array([[r_vel_est]])
 
    ev_md = ev['MD'].values.astype(float)
    ev_z  = ev['Z'].values.astype(float)
    ev_gr = gr_full.values.astype(float)[ev.index]
 
    out_vals = hw['TVT_input'].values.astype(float).copy()
    res      = np.empty(len(ev))
 
    prev_md = float(kmd[-1])
    prev_z  = float(kz[-1])
 
    for i in range(len(ev)):
        dm = max(ev_md[i] - prev_md, 1.0)
 
        # ── Predict step (linear) ─────────────────────────────────────────────
        F = np.array([[1., dm],
                      [0., 1.]])
 
        # FIX: Correct continuous-white-noise-acceleration (CWNA) Q matrix.
        #
        # For a constant-velocity model driven by acceleration noise w_a with
        # spectral density q_vel (units: (m/m)² / m = m/m²):
        #
        #   Q_vel = q_vel * | dm³/3   dm²/2 |
        #                   | dm²/2   dm    |
        #
        # This is the exact Van Loan discretisation of the Singer CWNA model.
        # The off-diagonal term q_vel * dm²/2 couples velocity noise into the
        # position channel, which the original diagonal form entirely missed.
        #
        # We then add independent position diffusion noise on the (0,0) entry
        # (the original q_pos * dm term), keeping that degree of freedom:
        #
        #   Q = Q_vel + diag([q_pos * dm, 0])
        #
        dm2 = dm  * dm
        dm3 = dm2 * dm
        Q = np.array([
            [q_vel * dm3 / 3.0 + q_pos * dm,   q_vel * dm2 / 2.0],
            [q_vel * dm2 / 2.0,                 q_vel * dm        ]
        ])
 
        x = F @ x
        P = F @ P @ F.T + Q
 
        # Clip predicted TVT to typewell range
        x[0] = float(np.clip(x[0], tw_tvt[0] - 100, tw_tvt[-1] + 100))
 
        # ── Update step 1: GR observation (non-linear → EKF linearisation) ────
        if not np.isnan(ev_gr[i]):
            # Numerical Jacobian of h(tvt) = interp(tw_gr, tvt)
            eps   = 0.5
            h_p   = float(np.interp(x[0] + eps, tw_tvt, tw_gr))
            h_m   = float(np.interp(x[0] - eps, tw_tvt, tw_gr))
            dh_dx = (h_p - h_m) / (2 * eps)               # ∂GR/∂TVT
            H_gr  = np.array([[dh_dx, 0.]])
 
            z_gr  = np.array([ev_gr[i]])
            h_hat = np.array([float(np.interp(x[0], tw_tvt, tw_gr))])
            y_gr  = z_gr - h_hat                           # innovation
 
            S_gr  = H_gr @ P @ H_gr.T + R_gr
            K_gr  = P @ H_gr.T @ np.linalg.inv(S_gr)      # Kalman gain
            x     = x + (K_gr @ y_gr).flatten()
            P     = (np.eye(2) - K_gr @ H_gr) @ P
 
        # ── Update step 2: velocity from inclination (linear; unchanged) ───────
        dz_step = ev_z[i] - prev_z
        dzdm    = dz_step / dm
        vel_exp = beta_kf * dzdm + icpt_kf        # expected velocity
        H_vel   = np.array([[0., 1.]])
        z_vel   = np.array([vel_exp])
        y_vel   = z_vel - H_vel @ x
        S_vel   = H_vel @ P @ H_vel.T + R_vel
        K_vel   = P @ H_vel.T @ np.linalg.inv(S_vel)
        x       = x + (K_vel @ y_vel).flatten()
        P       = (np.eye(2) - K_vel @ H_vel) @ P
 
        res[i]  = float(x[0])
        prev_md = ev_md[i]
        prev_z  = ev_z[i]
 
    out_vals[list(ev.index)] = res
    return out_vals.astype(np.float32)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# NEW: Rao-Blackwellised Particle Filter (RBPF)
# ─ Linear velocity state handled analytically by a KF per particle
# ─ Non-linear GR likelihood handled by particle weights
# ═══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
# FIX 1 — _rbpf_core
# ══════════════════════════════════════════════════════════════════════════════
 
@njit(cache=True)
def _rbpf_core(md_v, z_v, gr_v, gg, vmin, step,
               gs, ip, iv, iv_var,
               q_pos, q_vel, r_vel,
               N, RESAMP):
    """
    Rao-Blackwellised Particle Filter — Numba JIT inner loop.
 
    Each particle j maintains:
      pos[j]     : TVT pseudo-state  (pos = tvt + z,  non-linear part — sampled)
      vel_m[j]   : KF posterior mean of velocity given pos[j]  (linear — exact)
      vel_v[j]   : KF posterior variance of velocity
 
    CHANGES vs original
    -------------------
    The block previously commented
        "We don't have a direct vel observation here, so skip KF update"
    is REPLACED by the exact analytical Rao-Blackwell update derived below.
 
    Derivation (per particle j, per depth step i)
    -----------------------------------------------
    State model (scalar linear sub-system):
        vel_{t}  = vel_{t-1} + w_v,   w_v ~ N(0, q_vel * dm)
 
    Position sample (non-linear):
        pos_t ~ N(pos_{t-1} + v_pred * dm,  vel_v_pred * dm² + q_pos * dm)
 
    After the TVT sample pos[j] is drawn, we use the *realised* TVT increment
    as a pseudo-observation of velocity:
 
        z_v  = (pos_t - pos_{t-1}) / dm          [pseudo-observation]
        z_v  = vel_true + noise,  noise ~ N(0, R)
        R    = q_pos / dm                         [obs noise from pos diffusion]
 
    Standard scalar Kalman update:
        S  = P_pred + R
        K  = P_pred / S
        vel_m[j] = v_pred + K * (z_v - v_pred)
        vel_v[j] = (1 - K) * P_pred
    """
    # ── Initialise particles ──────────────────────────────────────────────────
    pos   = np.empty(N)
    vel_m = np.empty(N)   # conditional KF posterior mean
    vel_v = np.empty(N)   # conditional KF posterior variance
 
    # pos_prev[j] stores the previous step's position for z_v calculation
    pos_prev = np.empty(N)   # NEW: previous-step pos needed for innovation
 
    w = np.ones(N) / N
 
    for j in range(N):
        pos[j]      = ip   + 0.5  * np.random.randn()
        vel_m[j]    = iv   + 0.02 * np.random.randn()
        vel_v[j]    = iv_var
        pos_prev[j] = pos[j]   # initialise prev to starting position
 
    pts  = np.empty(len(md_v))
    stds = np.empty(len(md_v))
    pm   = md_v[0] - 1.
 
    for i in range(len(md_v)):
        dm = md_v[i] - pm
        dm = max(dm, 1.0)
 
        # ── KF predict for velocity (per particle) ────────────────────────────
        # Store predicted (prior) quantities before the TVT sample so we can
        # use them in the measurement update below.
        for j in range(N):
            # v_pred and P_pred are local scalars reused in the update step.
            # We write them back into vel_m / vel_v temporarily; the update
            # step below immediately corrects them with the KF gain.
            vel_v[j] += q_vel * dm          # P_pred = P_{t-1} + Q_vel * dm
 
        # ── Propagate TVT position: sample from p(tvt | v_pred, P_pred) ───────
        for j in range(N):
            pos_prev[j] = pos[j]            # save pos_{t-1} BEFORE updating
 
            # Spread of position draw:  vel uncertainty × dm² + intrinsic pos noise
            p_pos  = vel_v[j] * dm * dm + q_pos * dm
            pos[j] += vel_m[j] * dm + (p_pos ** 0.5) * np.random.randn()
 
            # Clip sampled TVT to typewell range
            tvt_j  = pos[j] - z_v[i]
            tvt_j  = max(tvt_j, vmin - 50.)
            tvt_j  = min(tvt_j, vmin + len(gg) * step + 50.)
            pos[j] = tvt_j + z_v[i]
 
        # ── GR likelihood weighting ───────────────────────────────────────────
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                tvt_j = pos[j] - z_v[i]
                eg    = _interp1(gg, tvt_j, vmin, step)
                d     = (gr_v[i] - eg) / gs
                lk    = max(np.exp(-0.5 * d * d) if d * d < 600. else 0., 1e-300)
                w[j] *= lk
                ws   += w[j]
            if ws > 0.:
                for j in range(N):
                    w[j] /= ws
            else:
                for j in range(N):
                    w[j] = 1. / N
 
        # ── KF measurement update for velocity (Rao-Blackwell step) ──────────
        #
        # FIX: this block replaces the original "skip KF update" comment.
        #
        # Pseudo-observation of velocity from the realised TVT increment:
        #   z_v_obs = (pos_t - pos_{t-1}) / dm
        #
        # Observation noise variance:
        #   R = q_pos / dm
        #   (position diffusion noise σ² = q_pos·dm divided by dm² gives
        #    velocity-equivalent noise σ² = q_pos/dm)
        #
        # Scalar Kalman update (no matrix inversion needed):
        #   S = P_pred + R
        #   K = P_pred / S           ∈ (0, 1)
        #   vel_m_post = v_pred + K * (z_v_obs - v_pred)
        #   vel_v_post = (1 - K) * P_pred
        #
        R_obs = q_pos / dm          # observation noise variance for z_v_obs
        for j in range(N):
            v_pred  = vel_m[j]      # prior mean  (already equals predicted mean
                                    # since we wrote v_pred into vel_m above)
            P_pred  = vel_v[j]      # prior variance (already updated by predict)
 
            # Pseudo-observation: realised velocity from incremental position
            z_v_obs = (pos[j] - pos_prev[j]) / dm
 
            S = P_pred + R_obs                  # innovation variance
            K = P_pred / S                      # Kalman gain  ∈ (0, 1)
 
            vel_m[j] = v_pred + K * (z_v_obs - v_pred)   # posterior mean
            vel_v[j] = (1.0 - K) * P_pred                # posterior variance
 
        # ── Resample if ESS drops below threshold ─────────────────────────────
        ne = 0.
        for j in range(N):
            ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            # Systematic resample — resample pos, vel_m, vel_v, pos_prev
            cum = np.zeros(N + 1)
            for j in range(N):
                cum[j + 1] = cum[j] + w[j]
            u0  = np.random.uniform(0., 1. / N)
            np2 = np.empty(N)
            nvm = np.empty(N)
            nvv = np.empty(N)
            npp = np.empty(N)   # NEW: resample pos_prev alongside pos
            ci  = 0
            for j in range(N):
                u = u0 + j / N
                while ci < N - 1 and cum[ci + 1] < u:
                    ci += 1
                np2[j] = pos[ci]      + 0.05  * np.random.randn()
                nvm[j] = vel_m[ci]    + 0.002 * np.random.randn()
                nvv[j] = vel_v[ci]
                npp[j] = pos_prev[ci]           # carry through without jitter
            for j in range(N):
                pos[j]      = np2[j]
                vel_m[j]    = nvm[j]
                vel_v[j]    = nvv[j]
                pos_prev[j] = npp[j]
                w[j]        = 1. / N
 
        # ── Weighted mean and std ─────────────────────────────────────────────
        tv = 0.
        for j in range(N):
            tv += w[j] * (pos[j] - z_v[i])
        pts[i] = tv
        va = 0.
        for j in range(N):
            va += w[j] * (pos[j] - z_v[i] - tv) ** 2
        stds[i] = va ** 0.5
        pm = md_v[i]
 
    return pts, stds
 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# NEW: Interacting Multiple Model (IMM) — KF + PF switching
# ═══════════════════════════════════════════════════════════════════════════════

def _gr_uncertainty(hw, tw_tvt, tw_gr, window=21):
    """
    Compute a per-depth-point GR uncertainty signal for the evaluation section.
    High uncertainty → prefer PF.  Low uncertainty → prefer KF.

    Uncertainty is estimated as the standard deviation of GR in a rolling window
    normalised by the overall GR std in the known section.
    """
    ev     = hw[hw['TVT_input'].isna()]
    kn     = hw[hw['TVT_input'].notna()]
    gr_all = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))

    # Rolling local std in eval section
    local_std = gr_all.rolling(window, center=True, min_periods=1).std().fillna(0)
    ev_local  = local_std.iloc[ev.index].values.astype(float)

    # Global std from known section as normaliser
    kn_gr = kn['GR'].dropna().values
    glob_std = float(np.std(kn_gr)) if len(kn_gr) > 5 else 20.0
    glob_std = max(glob_std, 1.0)

    return np.clip(ev_local / glob_std, 0.0, 3.0).astype(np.float32)


def run_imm_kf_pf(hw, tw_tvt, tw_gr,
                  N_rbpf=400,
                  sigma_thresh_low=0.4,
                  sigma_thresh_high=0.9):
    """
    Interacting Multiple Model filter.

    Two models run in parallel:
      M1: Extended Kalman Filter (reliable in smooth/linear zones)
      M2: Rao-Blackwellised Particle Filter (reliable in ambiguous/multimodal zones)

    At each depth step the mixing weight alpha(i) is set by the local GR uncertainty:
      sigma < sigma_thresh_low  → alpha = 0  (pure KF)
      sigma > sigma_thresh_high → alpha = 1  (pure RBPF)
      in between                → alpha = linear ramp

    IMM output: tvt_imm = (1 - alpha) * tvt_kf + alpha * tvt_rbpf

    Parameters
    ----------
    sigma_thresh_low  : below this normalised GR uncertainty → trust KF
    sigma_thresh_high : above this → trust RBPF

    Returns
    -------
    Full TVT array (known kept, eval filled by IMM)
    """
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    # Run KF
    kf_full  = run_kf_tvt(hw, tw_tvt, tw_gr)
    kf_eval  = kf_full[np.array(ev.index)]

    # Run RBPF
    rbpf_eval, rbpf_std = run_rbpf(hw, tw_tvt, tw_gr, N=N_rbpf)
    if len(rbpf_eval) == 0:
        return kf_full

    # Compute per-point GR uncertainty (mixing weight)
    sigma    = _gr_uncertainty(hw, tw_tvt, tw_gr)
    alpha    = np.clip(
        (sigma - sigma_thresh_low) / max(sigma_thresh_high - sigma_thresh_low, 1e-6),
        0.0, 1.0
    ).astype(np.float32)

    # IMM blend: alpha=0 → pure KF, alpha=1 → pure RBPF
    imm_eval = (1.0 - alpha) * kf_eval + alpha * rbpf_eval

    out_vals = hw['TVT_input'].values.astype(float).copy()
    out_vals[list(ev.index)] = imm_eval
    return out_vals.astype(np.float32)

In [ ]:
SEED=42
NCPU=min(4,multiprocessing.cpu_count())

FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

BEAMS=[
    (10,20.0,144.0,2,"cons"),
    (10, 8.0, 64.0,2,"loose"),
    ( 8,35.0,220.0,1,"vcons"),
    (10,14.0, 90.0,5,"sm5"),
    (20, 4.0, 36.0,3,"vloose"),
    (12,12.0,100.0,3,"mid"),
    (15,25.0,180.0,2,"stiff"),
]

PF_N=600; ANCC_N=600
PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    """Beam search ±2 delta, Numba JIT."""
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):            # ±2: TVT can go down
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=True)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IS,RP,RR,RESAMP):
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_

@njit(cache=True)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP):
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn()
        vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=(gr_sm_v[i]-es)/(gs*1.5)
                    ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

# ─── Python wrappers + grid helpers (unchanged) ───────────────────────────────

def _grid(tw_tvt, tw_gr, step=0.2):
    tmin  = float(tw_tvt.min());  tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)


def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw['TVT_input'].notna() & hw['GR'].notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(
        np.std(kn['GR'].values - np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)),
        PF_GR_SIG_MIN, PF_GR_SIG_MAX
    ))


def _nn(arr, v):
    i = int(np.searchsorted(arr, v, 'left'))
    if i >= len(arr): return len(arr) - 1
    if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v): return i - 1
    return i


def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r * 2 + 1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)


def beam_search_jit(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si   = _nn(tw_tvt, start_tvt)
    sgr  = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))
    return tw_tvt[path].astype(np.float32)


def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    kn = hw[hw['TVT_input'].notna()];  ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls   = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30)
    dt   = np.diff(tail['TVT_input'].values)
    dz   = np.diff(tail['Z'].values)
    dm   = np.diff(tail['MD'].values);  m = dm > 0
    ir   = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(
        ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
        ev['GR'].values.astype(np.float64), gg, gmin, gst,
        gs, ls, ir, N,
        ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP
    )
    return pts.astype(np.float32), std.astype(np.float32)


def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs   = _gr_sig(hw, tw_tvt, tw_gr)
    tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna  = hw[hw['TVT_input'].notna()];  ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k  = np.diff(kna['Z'].values);  dvt  = np.diff(kna['TVT_input'].values)
    dmd_k = np.diff(kna['MD'].values);  m2  = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2] / dmd_k[m2];  vt = dvt[m2] / dmd_k[m2]
        A  = np.column_stack([vz, np.ones_like(vz)]);  c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    else:
        beta, icpt, zsig = -1., 0., 0.1
    t2   = kna.tail(20);  dvt2 = np.diff(t2['TVT_input'].values);  dmd2 = np.diff(t2['MD'].values);  m3 = dmd2 > 0
    iv   = float(np.median(dvt2[m3] / dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gs2, _, _     = _grid(tw_tvt, tw_s)
    gr_sm = hw['GR'].rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(
        ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
        ev['GR'].values.astype(np.float64),
        gr_sm.loc[ev.index].values.astype(np.float64),
        gg, gs2, gmin, gst, gs, float(kna['TVT_input'].iloc[-1]), iv,
        beta, icpt, zsig, N,
        PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP
    )
    return pts.astype(np.float32), std.astype(np.float32)


# ── JIT warm-up (trigger compilation before parallel jobs) ────────────────────
_md = np.linspace(1, 50, 20, np.float64)
_z  = np.zeros(20, np.float64)
_gr = np.full(20, 50., np.float64)
_gg = np.linspace(45, 55, 100, np.float64)
_pf_ancc(_md, _z, _gr, _gg, 45., 0.1, 20., 50., 0., 8, 0.998, 0.002, 0.005, 0.3, 0.1, 0.001, 0.5)
_pf_z(_md, _z, _gr, _gr, _gg, _gg, 45., 0.1, 20., 50., 0., -1., 0., 0.1, 8, 0.993, 0.005, 0.01, 0.3, 0.2, 0.003, 0.5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
# Warm-up RBPF
_rbpf_core(_md, _z, _gr, _gg, 45., 0.1, 20., 50., 0., 0.01, 0.08, 0.001, 0.002, 8, 0.5)
print('JIT warm-up complete.')
_PIPELINE_START = time.perf_counter()
_WELL_TIMINGS   = {}          # wid -> dict of stage timings (seconds)
print(f'Pipeline started at {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

def robust_slope(x,y,w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def affine_cal(kgr,tw_at_k,min_pts=20):
    v=np.isfinite(kgr)&np.isfinite(tw_at_k)
    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

def seg_b_well(ktvt,kz,form_col):
    """Segment b_well: early/mid/late thirds + full prefix.
    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    # WLS (tail-upweighted)
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
    """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    # Score-weighted ensemble (NEW: softmax-weighted combination)
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

class FormationPlaneKNN:
    def __init__(self,well_ids,data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self,xy_q,self_wid=None,k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

# ─── Build spatial indices from training wells ────────────────────────────────
hw_paths   = sorted((CFG.dataset_path / 'train').glob('*__horizontal_well.csv'))
train_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
FI = FormationPlaneKNN(train_wids, CFG.dataset_path / 'train')
DI = DenseANCCImputer(train_wids,  CFG.dataset_path / 'train')

_FI = FI;  _DI = DI

ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10,  -5, -3, 0, 3,  5, 10, 20, 40], np.float32)
SC_OFFS   = np.array([-30, -15,  -8,  -4, -2, 0, 2,  4,  8, 15, 30], np.float32)
PF_OFFS   = np.array([-30, -15,  -8,  -4, -2, 0, 2,  4,  8, 15, 30], np.float32)

# ─── build_well: feature engineering (extended with KF/RBPF/IMM features) ─────

def build_well(hw_path, tw_path, is_train):
    global _FI, _DI
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path).sort_values('TVT')
    except:
        return None
    if is_train and 'TVT' not in hw.columns: return None
    kn = hw[hw['TVT_input'].notna()];  ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10:  return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt = tw['TVT'].to_numpy(np.float32)
    tw_gr  = tw['GR'].to_numpy(np.float32)
    if len(tw_tvt) < 3: return None

    _wt = {}
    _well_t0 = time.perf_counter() 

    # ── Original PF signals ───────────────────────────────────────────────────
    _t = time.perf_counter()
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    _wt['pf'] = time.perf_counter() - _t
    pf_use  = pf_a.astype(np.float32)
    std_use = std_a.astype(np.float32)
    has_z   = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))

    # ── NEW: KF signal ────────────────────────────────────────────────────────
    _t = time.perf_counter()
    try:
        kf_full   = run_kf_tvt(hw, tw_tvt, tw_gr)
        kf_eval   = kf_full[np.array(ev.index)]
        has_kf    = True
    except Exception:
        kf_eval   = pf_use.copy()
        has_kf    = False
    _wt['kf'] = time.perf_counter() - _t
    
    # ── NEW: RBPF signal ──────────────────────────────────────────────────────
    _t = time.perf_counter()
    try:
        rbpf_eval, rbpf_std = run_rbpf(hw, tw_tvt, tw_gr, N=400)
        has_rbpf  = len(rbpf_eval) == len(pf_a)
    except Exception:
        rbpf_eval = pf_use.copy()
        rbpf_std  = std_use.copy()
        has_rbpf  = False
    _wt['rbpf'] = time.perf_counter() - _t

    # ── NEW: IMM signal ───────────────────────────────────────────────────────
    _t = time.perf_counter()
    try:
        imm_full  = run_imm_kf_pf(hw, tw_tvt, tw_gr, N_rbpf=400)
        imm_eval  = imm_full[np.array(ev.index)]
        has_imm   = True
    except Exception:
        imm_eval  = pf_use.copy()
        has_imm   = False
    _wt['imm'] = time.perf_counter() - _t

    # ── NEW: per-point GR uncertainty (IMM mixing weight) ─────────────────────
    gr_unc = _gr_uncertainty(hw, tw_tvt, tw_gr)   # shape (n_eval,)

    lk        = kn.iloc[-1]
    last_tvt  = float(lk['TVT_input'])
    gr_full   = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr       = gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr       = gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # ── Beam search (7 configs) ───────────────────────────────────────────────
    _t = time.perf_counter()
    bpaths = {}
    for (bs, mc, es, r, tag) in BEAMS:
        bpaths[tag] = beam_search_jit(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.
    _wt['beam'] = time.perf_counter() - _t

    # ── Multi-scale NCC ───────────────────────────────────────────────────────
    _t = time.perf_counter()
    ktvt      = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8,  sc8s  = sc_res[0]
    sc15, sc15s = sc_res[1]
    sc25, sc25s = sc_res[2]
    sc_cons    = (sc8 + sc15 + sc25) / 3.
    sc_trust   = float(np.clip(len(kn) / 200., 0., 0.6))
    hyb_ref    = (1 - sc_trust) * beam_ref + sc_trust * sc_ens
    _wt['ncc'] = time.perf_counter() - _t

    tw_at_k    = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
    a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd        = kn['MD'].to_numpy(np.float32)
    kz         = kn['Z'].to_numpy(np.float32)
    pfx_rmse   = float(np.sqrt(np.mean((kgr - tw_at_k)**2)))
    slp_all    = robust_slope(kmd, ktvt)
    slp_50     = robust_slope(kmd[-50:], ktvt[-50:])
    slp_z      = robust_slope(kz, ktvt)

    _t = time.perf_counter()
    swid       = wid if is_train else None
    xy_ev      = ev[['X', 'Y']].to_numpy(np.float64)
    xy_kn      = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _     = _FI.impute(xy_kn, self_wid=swid)
    z_kn       = kn['Z'].to_numpy(np.float32)
    z_ev       = ev['Z'].to_numpy(np.float32)

    tvt_fs = {};  form_rmse = {};  form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f   = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fw  = (-z_ev + form_ev[:, fi2] + b_wls ).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'tvtF_{fn}']      = tvt_f
        tvt_fs[f'tvtFw_{fn}']     = tvt_fw
        tvt_fs[f'tvtF50_{fn}']    = tvt_f50
        tvt_fs[f'bw_{fn}']        = np.float32(b_full)
        tvt_fs[f'bww_{fn}']       = np.float32(b_wls)
        tvt_fs[f'bw50_{fn}']      = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}']  = np.float32(b_early)
        tvt_fs[f'bw_mid_{fn}']    = np.float32(b_mid)
        form_rmse[fn] = float(np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full))**2)))
        form_list.append(tvt_f)

    fs           = np.stack(form_list, 1)
    form_mean_d  = (fs.mean(1) - last_tvt).astype(np.float32)
    form_std_d   = fs.std(1).astype(np.float32)
    form_rng_d   = (fs.max(1) - fs.min(1)).astype(np.float32)

    d_ancc, d_std, d_dist   = _DI.impute(xy_ev, self_wid=swid)
    d_kn,   d_std_kn, _     = _DI.impute(xy_kn, self_wid=swid)
    b_vd   = ktvt + z_kn - d_kn
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    b_d    = float(np.median(b_vd))
    tvt_dense   = (-z_ev + d_ancc + b_d ).astype(np.float32)
    tvt_densew  = (-z_ev + d_ancc + b_dw).astype(np.float32)
    tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn
    d_rmse = float(np.sqrt(np.mean(res_kn**2)))
    d_bias = float(np.mean(res_kn))
    d_nb_std = float(np.mean(d_std_kn))
    _wt['spatial'] = time.perf_counter() - _t

    all_sigs = ([pf_use] + [p for p in bpaths.values()] +
                [sc8, sc15, sc25, sc_ens, tvt_fs['tvtF_ANCC'], tvt_dense])
    sig_mat  = np.stack(all_sigs, 1)
    sig_std  = sig_mat.std(1).astype(np.float32)
    sig_mean = (sig_mat.mean(1) - last_tvt).astype(np.float32)

    gr_s  = pd.Series(gr_full.values);  rolls = {}
    for w_r in [5, 21, 51, 101]:
        r_obj = gr_s.rolling(w_r, center=True, min_periods=1)
        rolls[f'grm{w_r}'] = r_obj.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w_r}'] = r_obj.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        rolls[f'glag{lag}']  = gr_s.shift(lag ).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}'] = gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1  = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2  = gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg = np.sqrt(np.maximum(
        (gr_s**2).rolling(21, center=True, min_periods=1).mean(), 0.
    )).iloc[ev.index].values.astype(np.float32)

    hmd        = ev['MD'].to_numpy(np.float32)
    md_since   = hmd - float(lk['MD'])
    slp_b_all  = (last_tvt + slp_all * md_since).astype(np.float32)
    slp_b_50   = (last_tvt + slp_50  * md_since).astype(np.float32)

    mdd   = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)

    nh   = len(ev)
    frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
    def sc(v): return np.full(nh, np.float32(v), np.float32)

    feats = {
        'well': wid,
        'id':   [f'{wid}_{i}' for i in ev.index],
        'last_known_tvt':     sc(last_tvt),
        # ── Original PF features ──────────────────────────────────────────────
        'pf_ancc':            pf_use,
        'pf_ancc_std':        std_use,
        'pf_ancc_delta':      (pf_use - last_tvt).astype(np.float32),
        'pf_z':               (pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta':         ((pf_z - last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z':            ((pf_use - pf_z.astype(np.float32)) if has_z else sc(0.)),
        # ── NEW: KF features ──────────────────────────────────────────────────
        'kf_tvt':             kf_eval.astype(np.float32),
        'kf_delta':           (kf_eval - last_tvt).astype(np.float32),
        'pf_vs_kf':           (pf_use - kf_eval).astype(np.float32),
        # ── NEW: RBPF features ────────────────────────────────────────────────
        'rbpf_tvt':           (rbpf_eval.astype(np.float32) if has_rbpf else pf_use),
        'rbpf_std':           (rbpf_std.astype(np.float32)  if has_rbpf else std_use),
        'rbpf_delta':         ((rbpf_eval - last_tvt).astype(np.float32) if has_rbpf else sc(0.)),
        'rbpf_vs_pf':         ((rbpf_eval - pf_use).astype(np.float32)  if has_rbpf else sc(0.)),
        # ── NEW: IMM features ─────────────────────────────────────────────────
        'imm_tvt':            imm_eval.astype(np.float32),
        'imm_delta':          (imm_eval - last_tvt).astype(np.float32),
        'imm_vs_pf':          (imm_eval - pf_use).astype(np.float32),
        'imm_vs_kf':          (imm_eval - kf_eval).astype(np.float32),
        # ── NEW: GR uncertainty (IMM mixing weight as a feature) ──────────────
        'gr_uncertainty':     gr_unc,
        # ── Beam features ─────────────────────────────────────────────────────
        **{f'beam_{t}_d': (p - np.float32(last_tvt)).astype(np.float32) for t, p in bpaths.items()},
        'beam_mean_d':        np.stack([(p - last_tvt) for p in bpaths.values()], 1).mean(1).astype(np.float32),
        'beam_std_d':         np.stack([(p - last_tvt) for p in bpaths.values()], 1).std(1).astype(np.float32),
        'beam_med_d':         np.median(np.stack([(p - last_tvt) for p in bpaths.values()], 1), 1).astype(np.float32),
        # ── NCC features ──────────────────────────────────────────────────────
        'sc8_d':              (sc8  - np.float32(last_tvt)).astype(np.float32),
        'sc8_sc':             sc8s,
        'sc15_d':             (sc15 - np.float32(last_tvt)).astype(np.float32),
        'sc15_sc':            sc15s,
        'sc25_d':             (sc25 - np.float32(last_tvt)).astype(np.float32),
        'sc25_sc':            sc25s,
        'sc_cons_d':          (sc_cons - np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d':           (sc_ens  - np.float32(last_tvt)).astype(np.float32),
        'sc_trust':           sc(sc_trust),
        'hyb_d':              (hyb_ref - np.float32(last_tvt)).astype(np.float32),
        'sig_std':            sig_std,
        'sig_mean_d':         sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}':  sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d':        form_mean_d,
        'form_std_d':         form_std_d,
        'form_rng_d':         form_rng_d,
        'spatial_ancc_d':     (form_ev[:, 0] - np.float32(np.interp(last_tvt, tw_tvt, tw_gr))),
        'spatial_knn_dist':   knn_d,
        'dense_ancc':         d_ancc,
        'dense_std':          d_std,
        'dense_dist':         d_dist,
        'tvt_dense_d':        (tvt_dense   - last_tvt).astype(np.float32),
        'tvt_densew_d':       (tvt_densew  - last_tvt).astype(np.float32),
        'tvt_dense50_d':      (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse':         sc(d_rmse),
        'dense_bias':         sc(d_bias),
        'dense_nb_std':       sc(d_nb_std),
        'pf_vs_spatial':      (pf_use - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense':        (pf_use - tvt_dense).astype(np.float32),
        'spatial_vs_dense':   (tvt_fs['tvtF_ANCC'] - tvt_dense).astype(np.float32),
        'beam_vs_spatial':    (bpaths['cons'] - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam':         (sc_ens - bpaths['cons']).astype(np.float32),
        'cal_a':              sc(a_cal),
        'cal_b':              sc(b_cal),
        'pfx_rmse':           sc(pfx_rmse),
        'known_len':          sc(len(kn)),
        'eval_len':           sc(nh),
        'slp_all':            sc(slp_all),
        'slp_50':             sc(slp_50),
        'slp_z':              sc(slp_z),
        'slp_b_d_all':        (slp_b_all - last_tvt).astype(np.float32),
        'slp_b_d_50':         (slp_b_50  - last_tvt).astype(np.float32),
        'ktvt_range':         sc(float(np.ptp(ktvt))),
        'ktvt_std':           sc(float(ktvt.std())),
        'md_since':           md_since,
        'frac':               frac,
        'frac2':              frac**2,
        'sqrt_frac':          np.sqrt(frac),
        'z':                  z_ev,
        'dx':                 (ev['X'] - float(lk['X'])).to_numpy(np.float32),
        'dy':                 (ev['Y'] - float(lk['Y'])).to_numpy(np.float32),
        'dz':                 (z_ev - float(lk['Z'])).astype(np.float32),
        'dxy':                np.sqrt((ev['X'] - float(lk['X']))**2 + (ev['Y'] - float(lk['Y']))**2).to_numpy(np.float32),
        'dzdmd':              dzdmd,
        'dxdmd':              dxdmd,
        'dydmd':              dydmd,
        'gr':                 hgr,
        'gr_d1':              gr_d1,
        'gr_d2':              gr_d2,
        'gr_env':             gr_env,
        'gr_nrg':             gr_nrg,
        'gr_vs_tw_anc':       hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)),
        'gr_vs_slp_all':      hgr - np.interp(slp_b_all, tw_tvt, tw_gr).astype(np.float32),
        **{f'tda{int(o)}':    hgr - np.float32(np.interp(last_tvt + o, tw_tvt, tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}':   hgr - np.interp(beam_ref + o, tw_tvt, tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}':   hgr - np.interp(sc_ens   + o, tw_tvt, tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}':   hgr - np.interp(pf_use   + o, tw_tvt, tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range':           sc(float(np.ptp(tw_tvt))),
        'tw_gr_mean':         sc(float(tw_gr.mean())),
    }
    for k, v in rolls.items(): feats[k] = v

    result = pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target'] = (ev['TVT'].to_numpy(np.float32) - np.float32(last_tvt))
        
    _wt['total']  = time.perf_counter() - _well_t0
    _wt['n_eval'] = len(ev)
    _wt['n_known'] = len(kn)
    _WELL_TIMINGS[wid] = _wt
    stages = ['pf', 'kf', 'rbpf', 'imm', 'beam', 'ncc', 'spatial']
    stage_str = '  '.join(
        f'{k}={_wt.get(k, 0):.2f}s' for k in stages
    )
    print(
        f'  [{wid}] total={_wt["total"]:.2f}s | {stage_str} | '
        f'n_known={_wt["n_known"]}  n_eval={_wt["n_eval"]}'
    )
    return result

def build_dataset(paths, is_train, label):
    args = [
        (str(p),
         str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'),
         is_train)
        for p in paths
        if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()
    ]
    t0  = time.time()
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=3)(
        delayed(build_well)(hp, tp, it) for hp, tp, it in args
    )
    parts = [r for r in res if r is not None]
    print(f'{label}: {len(parts)} wells built in {time.time()-t0:.0f}s')
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def build_dataset(paths, is_train, label):
    args = [
        (str(p),
         str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'),
         is_train)
        for p in paths
        if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()
    ]
    t0  = time.perf_counter()
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=3)(
        delayed(build_well)(hp, tp, it) for hp, tp, it in args
    )
    parts = [r for r in res if r is not None]
    elapsed = time.perf_counter() - t0

    # ── Per-dataset aggregate timing summary ──────────────────────────────────
    built_wids   = [p.stem.replace('__horizontal_well', '') for p in paths]
    timed_wids   = [w for w in built_wids if w in _WELL_TIMINGS]
    if timed_wids:
        stages = ['pf', 'kf', 'rbpf', 'imm', 'beam', 'ncc', 'spatial', 'total']
        rows   = [{s: _WELL_TIMINGS[w].get(s, 0) for s in stages} | {'wid': w}
                  for w in timed_wids]
        timing_df = pd.DataFrame(rows).set_index('wid')
        stage_means = timing_df[stages].mean()

        print(f'\n{"-" * 70}')
        print(f'{label.upper()} DATASET — TIMING SUMMARY  ({len(parts)} wells  |  {elapsed:.1f}s total)')
        print(f'{"-" * 70}')
        print(f'  {"Well":<14}  {"total":>7}  {"pf":>7}  {"kf":>7}  {"rbpf":>7}  {"imm":>7}  {"beam":>7}  {"ncc":>7}  {"spatial":>8}  n_eval')
        print(f'  {"-"*14}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*8}  ------')
        for w in timed_wids:
            t = _WELL_TIMINGS[w]
            print(
                f'  {w:<14}  {t.get("total",0):>6.2f}s'\
                f'  {t.get("pf",0):>6.2f}s'\
                f'  {t.get("kf",0):>6.2f}s'\
                f'  {t.get("rbpf",0):>6.2f}s'\
                f'  {t.get("imm",0):>6.2f}s'\
                f'  {t.get("beam",0):>6.2f}s'\
                f'  {t.get("ncc",0):>6.2f}s'\
                f'  {t.get("spatial",0):>7.2f}s'\
                f'  {t.get("n_eval",0):>6d}'\
            )
        print(f'  {"-"*14}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*7}  {"-"*8}')
        print(
            f'  {"MEAN":<14}  {stage_means["total"]:>6.2f}s'\
            f'  {stage_means["pf"]:>6.2f}s'\
            f'  {stage_means["kf"]:>6.2f}s'\
            f'  {stage_means["rbpf"]:>6.2f}s'\
            f'  {stage_means["imm"]:>6.2f}s'\
            f'  {stage_means["beam"]:>6.2f}s'\
            f'  {stage_means["ncc"]:>6.2f}s'\
            f'  {stage_means["spatial"]:>7.2f}s'\
        )
        print(f'{"-" * 70}')
        slowest = timing_df['total'].idxmax()
        fastest = timing_df['total'].idxmin()
        print(f'  Slowest well : {slowest}  ({timing_df.loc[slowest, "total"]:.2f}s)')
        print(f'  Fastest well : {fastest}  ({timing_df.loc[fastest, "total"]:.2f}s)')
        print(f'  Dataset wall time: {str(timedelta(seconds=int(elapsed)))}  ({elapsed:.1f}s)')
        print(f'{"-" * 70}\n')

    print(f'{label}: {len(parts)} wells built in {elapsed:.0f}s')
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

In [ ]:
# ─── Load / build datasets ────────────────────────────────────────────────────
_t_data = time.perf_counter()

if (CFG.artifacts_path / 'data' / 'train.csv').exists():
    train_df = pd.read_csv(CFG.artifacts_path / 'data' / 'train.csv', low_memory=False)
    # Recompute KF/RBPF/IMM columns if not already present (backward compat)
    for new_col in ['kf_delta', 'rbpf_delta', 'imm_delta', 'gr_uncertainty']:
        if new_col not in train_df.columns:
            print(f'Column {new_col} not in cached CSV — rebuilding train dataset...')
            train_paths = sorted((CFG.dataset_path / 'train').glob('*__horizontal_well.csv'))
            train_df    = build_dataset(train_paths, is_train=True, label='train')
            break
else:
    train_paths = sorted((CFG.dataset_path / 'train').glob('*__horizontal_well.csv'))
    train_df    = build_dataset(train_paths, is_train=True, label='train')

test_paths = sorted((CFG.dataset_path / 'test').glob('*__horizontal_well.csv'))
test_df    = build_dataset(test_paths, is_train=False, label='test')

features = [c for c in train_df.columns if c not in {'well', 'id', 'target'}]

X      = train_df[features]
y      = train_df['target']
g      = train_df['well']
X_test = test_df[features]

_SECTION_TIMES = {}   # stage-level wall-clock registry for the whole pipeline
_SECTION_TIMES['data_loading'] = time.perf_counter() - _t_data
print(f'\n[TIMER] Data loading & feature engineering: {_SECTION_TIMES["data_loading"]:.1f}s')

# 3. Training

In [ ]:
lgb_params = [
    dict(
        boosting_type='gbdt', num_leaves=255, min_child_samples=15,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=3.0, reg_alpha=0.05, objective='regression',
        verbose=-1, n_jobs=-1, device_type='gpu', gpu_use_dp=False,
        max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123
    ),
    dict(
        n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913,
        subsample=0.47437582748953966, num_leaves=64,
        reg_lambda=95.75401894533888, n_estimators=10000, random_state=0,
        boosting_type='gbdt', learning_rate=0.00934485794382918,
        colsample_bytree=0.39283351290380497,
        min_child_weight=0.24081152127177283, min_child_samples=40, device='gpu',
    ),
    dict(
        n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913,
        subsample=0.47437582748953966, num_leaves=64,
        reg_lambda=95.75401894533888, n_estimators=10000, random_state=29,
        boosting_type='gbdt', learning_rate=0.00934485794382918,
        colsample_bytree=0.39283351290380497,
        min_child_weight=0.24081152127177283, min_child_samples=40, device='gpu',
    ),
    dict(
        boosting_type='gbdt', 
        num_leaves=128,             # Changed from 64/255 to force different tree structures
        min_child_samples=20,       
        subsample=0.7, 
        colsample_bytree=0.6,       # Lower column sampling to force variety
        reg_lambda=10.0, 
        reg_alpha=1.0, 
        objective='regression',
        verbose=-1, 
        n_jobs=-1, 
        device_type='gpu', 
        learning_rate=0.015, 
        n_estimators=7000, 
        seed=999                    # Completely different seed
    )
]

cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15,
         border_count=254, loss_function='RMSE', task_type='GPU', devices='0',
         od_type='Iter', od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15,
         border_count=254, loss_function='RMSE', task_type='GPU', devices='0',
         od_type='Iter', od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
]

ridge_params = {
    'random_state': 42,
    'alpha':        1.6602834637650032,
    'tol':          0.0005030247295617308,
    'positive':     True,
    'fit_intercept': True,
}

# Lasso params — L1 regularisation; drives weak/redundant base-model weights to zero.
# A high alpha prunes base models aggressively; positive=True mirrors Ridge convention.
lasso_params = {
    'alpha':         0.001,          # Relatively small: boosters are correlated
    'positive':      True,           # Non-negative weights (blend, not subtract)
    'fit_intercept': True,
    'tol':           1e-4,
    'max_iter':      5000,
}

# ElasticNet params — L1 + L2 mixture. l1_ratio=0 → Ridge, l1_ratio=1 → Lasso.
# A balanced ratio gives automatic feature selection (L1) + stability (L2).
elasticnet_params = {
    'alpha':         0.001,
    'l1_ratio':      0.5,            # Equal L1 / L2 blend (tunable)
    'positive':      True,
    'fit_intercept': True,
    'tol':           1e-4,
    'max_iter':      5000,
}

# Stacking meta-learner params — ElasticNet trained on OOF predictions of
# Ridge, Lasso and ElasticNet; gives a second-level regularised blend.
stacking_meta_params = {
    'alpha':         0.0005,
    'l1_ratio':      0.3,            # Slightly L2-heavy for stable stacking
    'positive':      True,
    'fit_intercept': True,
    'tol':           1e-4,
    'max_iter':      5000,
}

# Post-processing params — w_pf now blends towards IMM instead of raw PF
pp_params = {
    'alpha': 1.0,
    'tau':   85,
    'w_pf':  0.09,   # semantically: weight towards IMM signal (= best physics)
}

oof_preds     = {}
test_preds    = {}
overall_scores = {}
fold_scores   = {}

## 3.1 LightGBM

In [ ]:
_t_lgb = time.perf_counter()

for i, params in enumerate(lgb_params):
    save_path = f'models/lightgbm-new-{i+1}'
    if (CFG.models_path / save_path).exists():
        print(f'Loading lightgbm-{i+1} from disk...')
        trainer_paths = (CFG.models_path / save_path).glob('*.pkl')
        trainer = joblib.load(list(trainer_paths)[0])
        print(f'Loaded lightgbm-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n')
    else:
        trainer = Trainer(
            estimator=LGBMRegressor(**params),
            task='regression', metric=CFG.metric,
            cv=CFG.cv, cv_args={'groups': g},
            use_early_stopping=True, verbose=True,
            save=True, save_path=save_path
        )
        trainer.fit(
            X, y,
            fit_args={
                'eval_metric': 'rmse',
                'callbacks': [log_evaluation(period=250), early_stopping(stopping_rounds=250)]
            }
        )
        print('\n\n')

    oof_preds[f'lightgbm-{i+1}']     = trainer.oof_preds
    test_preds[f'lightgbm-{i+1}']    = trainer.predict(X_test)
    overall_scores[f'lightgbm-{i+1}'] = trainer.overall_score
    fold_scores[f'lightgbm-{i+1}']   = trainer.fold_scores

_SECTION_TIMES['lightgbm_training'] = time.perf_counter() - _t_lgb
print(f'\n[TIMER] LightGBM training: {_SECTION_TIMES["lightgbm_training"]:.1f}s')

## 3.2 CatBoost

In [ ]:
_t_cb = time.perf_counter()

for i, params in enumerate(cb_params):
    save_path = f'models/catboost-new-{i+1}'
    if (CFG.models_path / save_path).exists():
        print(f'Loading catboost-{i+1} from disk...')
        trainer_paths = (CFG.models_path / save_path).glob('*.pkl')
        trainer = joblib.load(list(trainer_paths)[0])
        print(f'Loaded catboost-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n')
    else:
        trainer = Trainer(
            estimator=CatBoostRegressor(**params),
            task='regression', metric=CFG.metric,
            cv=CFG.cv, cv_args={'groups': g},
            use_early_stopping=True, verbose=True,
            save=True, save_path=save_path
        )
        trainer.fit(
            X, y,
            fit_args={'verbose': 250, 'early_stopping_rounds': 250, 'use_best_model': True}
        )
        print('\n\n')

    oof_preds[f'catboost-{i+1}']     = trainer.oof_preds
    test_preds[f'catboost-{i+1}']    = trainer.predict(X_test)
    overall_scores[f'catboost-{i+1}'] = trainer.overall_score
    fold_scores[f'catboost-{i+1}']   = trainer.fold_scores

_SECTION_TIMES['catboost_training'] = time.perf_counter() - _t_cb
print(f'\n[TIMER] CatBoost training: {_SECTION_TIMES["catboost_training"]:.1f}s')

# 4. Ensembling with Ridge + Lasso + ElasticNet

In [ ]:
_t_ens = time.perf_counter()

oof_preds  = pd.DataFrame(oof_preds)
test_preds = pd.DataFrame(test_preds)

# ─── Layer-1 meta-learners: Ridge, Lasso, ElasticNet ─────────────────────────
# All three are trained on the same OOF predictions from LightGBM + CatBoost.
# Each brings a different regularisation bias:
#   Ridge   — shrinks all weights proportionally (L2)
#   Lasso   — zeros out weak base-model contributions (L1)
#   ElasticNet — L1+L2 balance: selection + stability

# 4a. Ridge (unchanged from original — baseline meta-learner)
ridge_trainer = Trainer(
    Ridge(**ridge_params),
    task='regression', metric=CFG.metric,
    cv=CFG.cv, cv_args={'groups': g},
    verbose=True
)
ridge_trainer.fit(oof_preds, y)

ridge_oof_preds  = ridge_trainer.oof_preds
ridge_test_preds = ridge_trainer.predict(test_preds)

overall_scores['ridge'] = ridge_trainer.overall_score
fold_scores['ridge']    = ridge_trainer.fold_scores
print(f'Ridge RMSE: {ridge_trainer.overall_score:.4f}')

# 4b. Lasso — L1 regularisation; sparse weights, prunes correlated models
lasso_trainer = Trainer(
    Lasso(**lasso_params),
    task='regression', metric=CFG.metric,
    cv=CFG.cv, cv_args={'groups': g},
    verbose=True
)
lasso_trainer.fit(oof_preds, y)

lasso_oof_preds  = lasso_trainer.oof_preds
lasso_test_preds = lasso_trainer.predict(test_preds)

overall_scores['lasso'] = lasso_trainer.overall_score
fold_scores['lasso']    = lasso_trainer.fold_scores
print(f'Lasso RMSE: {lasso_trainer.overall_score:.4f}')

# 4c. ElasticNet — equal L1+L2; combines sparsity and shrinkage
enet_trainer = Trainer(
    ElasticNet(**elasticnet_params),
    task='regression', metric=CFG.metric,
    cv=CFG.cv, cv_args={'groups': g},
    verbose=True
)
enet_trainer.fit(oof_preds, y)

enet_oof_preds  = enet_trainer.oof_preds
enet_test_preds = enet_trainer.predict(test_preds)

overall_scores['elasticnet'] = enet_trainer.overall_score
fold_scores['elasticnet']    = enet_trainer.fold_scores
print(f'ElasticNet RMSE: {enet_trainer.overall_score:.4f}')

# ─── Layer-2 stacking: ElasticNet meta-learner on L1 OOF predictions ─────────
# The three L1 regressors above produce their own OOF predictions; a second-level
# ElasticNet learns the optimal linear combination, removing any remaining
# correlated bias between Ridge / Lasso / ElasticNet.
l1_oof_preds = pd.DataFrame({
    'ridge':      ridge_oof_preds,
    'lasso':      lasso_oof_preds,
    'elasticnet': enet_oof_preds,
})
l1_test_preds = pd.DataFrame({
    'ridge':      ridge_test_preds,
    'lasso':      lasso_test_preds,
    'elasticnet': enet_test_preds,
})

stacking_trainer = Trainer(
    ElasticNet(**stacking_meta_params),
    task='regression', metric=CFG.metric,
    cv=CFG.cv, cv_args={'groups': g},
    verbose=True
)
stacking_trainer.fit(l1_oof_preds, y)

stacking_oof_preds  = stacking_trainer.oof_preds
stacking_test_preds = stacking_trainer.predict(l1_test_preds)

overall_scores['stacking (ridge+lasso+enet)'] = stacking_trainer.overall_score
fold_scores['stacking (ridge+lasso+enet)']    = stacking_trainer.fold_scores
print(f'Stacking (Ridge+Lasso+ElasticNet) RMSE: {stacking_trainer.overall_score:.4f}')

# ─── Unified alias used downstream ────────────────────────────────────────────
# Post-processing and inference reference `ridge_oof_preds` / `ridge_test_preds`.
# We redirect these to the stacking output so all downstream cells work unchanged.
# The individual meta-learner predictions are also retained for diagnostics.
ridge_oof_preds  = stacking_oof_preds
ridge_test_preds = stacking_test_preds
print('\nMeta-learner outputs redirected to stacking predictions for downstream use.')
_SECTION_TIMES['ensembling'] = time.perf_counter() - _t_ens
print(f'\n[TIMER] Ensembling (Ridge+Lasso+ElasticNet stacking): {_SECTION_TIMES["ensembling"]:.1f}s')

# 5. Postprocessing

In [ ]:
_t_pp = time.perf_counter()

def apply_pp(df, md, pd_, alpha, tau, w_pf):
    """
    Post-processing: blend stacking-ensemble delta (Ridge+Lasso+ElasticNet)
    with physics signal (IMM if available, else pf_ancc), apply exponential ramp
    for continuity at the join point.
    pd_ can be either pf_ancc_delta or imm_delta (both in same units).
    """
    d = md * (1 - w_pf) + pd_ * w_pf
    if tau:
        d *= (1. - np.exp(-np.maximum(df['md_since'].values, 0.) / tau))
    return d * alpha


def sg_smooth(df, col, sg_w=17, sg_p=3):
    df = df.copy()
    for _, g_well in df.groupby('well', sort=False):
        v  = g_well[col].values
        n  = len(v)
        wl = min(sg_w, n)
        if wl % 2 == 0: wl -= 1
        if wl >= sg_p + 2: v = savgol_filter(v, wl, sg_p)
        df.loc[g_well.index, col] = v
    return df


base   = train_df['last_known_tvt'].values
ytrue  = y.values + base

# Use IMM delta for post-processing if available, else fall back to pf_ancc
if 'imm_delta' in train_df.columns:
    phys_oof = train_df['imm_delta'].values
else:
    phys_oof = train_df['pf_ancc'].values - base

d              = apply_pp(train_df, ridge_oof_preds, phys_oof, **pp_params)
stacking_score = root_mean_squared_error(ytrue, base + d)

overall_scores['stacking (pp)'] = stacking_score
fold_scores['stacking (pp)']    = [stacking_score] * CFG.n_splits
print(f'Stacking (Ridge+Lasso+ElasticNet) + PP RMSE: {stacking_score:.4f}')
_SECTION_TIMES['postprocessing'] = time.perf_counter() - _t_pp
print(f'[TIMER] Post-processing: {_SECTION_TIMES["postprocessing"]:.1f}s')

# 6. Inference

## 6.1 Ridge + Lasso + ElasticNet

In [ ]:
_t_inf1 = time.perf_counter()

test_df2 = test_df.copy()

# Use IMM delta for pp if available
if 'imm_delta' in test_df2.columns:
    phys_test = test_df2['imm_delta'].values
else:
    phys_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values

test_df2['pred'] = (test_df2['last_known_tvt'].values
                    + apply_pp(test_df2, ridge_test_preds, phys_test, **pp_params))
test_df2 = sg_smooth(test_df2, 'pred')

sample_sub = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
sub_1 = (
    sample_sub[['id']].merge(
        test_df2[['id', 'pred']].rename(columns={'pred': 'tvt'}),
        on='id', how='left'
    )
)
sub_1['tvt'] = sub_1['tvt'].fillna(
    float(train_df['last_known_tvt'].mean() + train_df['target'].mean())
)

_SECTION_TIMES['ml_inference'] = time.perf_counter() - _t_inf1
print(f'[TIMER] ML inference (sub_1): {_SECTION_TIMES["ml_inference"]:.1f}s')

sub_1

## 6.2 Heuristic model

In [ ]:
sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

train_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'train' / '*__horizontal_well.csv')))
train_wells    = [os.path.basename(f).split('__')[0] for f in train_hw_files]

test_hw_files  = sorted(glob.glob(str(CFG.dataset_path / 'test' / '*__horizontal_well.csv')))
test_wells     = [os.path.basename(f).split('__')[0] for f in test_hw_files]

rows = []
_heuristic_well_times = []   # list of dicts: wid, total, pf_ens, beam, selector
_t_heuristic = time.perf_counter()

for i, wid in enumerate(test_wells):
    _well_start = time.perf_counter()
    print(f'\nProcessing {i + 1}/{len(test_wells)}: {wid}...')
    hw_te, tw_te = load_well(wid, 'test')

    tvt_phys = None
    hw_tr    = None
    tw_tr    = None

    # ── Physical model for wells visible in train ─────────────────────────────
    _t = time.perf_counter()
    if wid in train_wells:
        try:
            hw_tr, tw_tr = load_well(wid, 'train')
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print(f'  Physical model (contacts) OK')
        except Exception as e:
            print(f'  Physical model failed: {e}')
            tvt_phys = None
    _t_phys = time.perf_counter() - _t

    selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)
    tw_ref = tw_tr if tw_tr is not None else tw_te
    tw_s   = tw_ref.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    # ── 150-seed likelihood-weighted PF ensemble (+ KF + IMM inside) ──────────
    _t = time.perf_counter()
    try:
        pf_by_scale = run_pf_lik_ensemble_scales(
            hw_te, tw_ref, n_particles=600, n_seeds=150
        )
        tvt_pf = pf_by_scale.get('pf_scale_8', list(pf_by_scale.values())[0])
        print(f'  PF+KF+IMM ensemble OK  scales={SELECTOR_SCALES}')
    except Exception as e:
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf     = hw_te['TVT_input'].fillna(last_val).values.astype(float)
        pf_by_scale = {f'pf_scale_{s:g}': tvt_pf.copy() for s in SELECTOR_SCALES}
        pf_by_scale.update({f'imm_scale_{s:g}': tvt_pf.copy() for s in SELECTOR_SCALES})
        pf_by_scale['kf'] = tvt_pf.copy()
    _t_pf_ens = time.perf_counter() - _t

    # ── Beam search ensemble ───────────────────────────────────────────────────
    _t = time.perf_counter()
    try:
        tvt_beam = run_beam_ensemble(hw_te, tw_ref)
        print(f'  Beam 14-config ensemble OK')
    except Exception as e:
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()
    _t_beam = time.perf_counter() - _t

    # ── Selector blending (now supports kf / imm variants) ────────────────────
    _t = time.perf_counter()
    last_known     = hw_te['TVT_input'].dropna()
    last_known_tvt = (float(last_known.iloc[-1]) if len(last_known) > 0
                      else float(np.nanmean(tvt_pf)))
    tvt_selector = apply_selector_variant(
        selector_variant, pf_by_scale, tvt_beam, last_known_tvt
    )
    _t_selector = time.perf_counter() - _t

    print(
        f'  Selector code={selector_code} variant={selector_variant} '\
        f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'\
    )

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx    = int(row['row_idx'])
        tvt_val = float(tvt_phys.iloc[ridx]) if tvt_phys is not None else float(tvt_selector[ridx])
        rows.append({'id': row['id'], 'tvt': tvt_val})
    print(f'  Added {len(ws)} rows')

    # ── Per-well heuristic timing record ──────────────────────────────────────
    _well_total = time.perf_counter() - _well_start
    _heuristic_well_times.append({
        'wid':      wid,
        'total':    _well_total,
        'phys':     _t_phys,
        'pf_ens':   _t_pf_ens,
        'beam':     _t_beam,
        'selector': _t_selector,
        'n_eval':   int(selector_n_eval),
    })
    print(
        f'  [WELL TIMER] {wid}: total={_well_total:.2f}s  '\
        f'phys={_t_phys:.2f}s  pf_ens={_t_pf_ens:.2f}s  '\
        f'beam={_t_beam:.2f}s  selector={_t_selector:.3f}s'\
    )

sub_2 = pd.DataFrame(rows)

# ── Heuristic pipeline timing summary ─────────────────────────────────────────
_t_heuristic_total = time.perf_counter() - _t_heuristic
_SECTION_TIMES['heuristic_inference'] = _t_heuristic_total

_hw_df = pd.DataFrame(_heuristic_well_times)
if len(_hw_df):
    print(f'\n{"-" * 72}')
    print(f'HEURISTIC INFERENCE — PER-WELL TIMING SUMMARY  ({len(_hw_df)} wells  |  {_t_heuristic_total:.1f}s total)')
    print(f'{"-" * 72}')
    print(f'  {"Well":<14}  {"total":>7}  {"phys":>7}  {"pf_ens":>8}  {"beam":>7}  {"selector":>9}  {"n_eval":>7}')
    print(f'  {"-"*14}  {"-"*7}  {"-"*7}  {"-"*8}  {"-"*7}  {"-"*9}  {"-"*7}')
    for rec in _heuristic_well_times:
        print(
            f'  {rec["wid"]:<14}  {rec["total"]:>6.2f}s'\
            f'  {rec["phys"]:>6.2f}s'\
            f'  {rec["pf_ens"]:>7.2f}s'\
            f'  {rec["beam"]:>6.2f}s'\
            f'  {rec["selector"]:>8.3f}s'\
            f'  {rec["n_eval"]:>7d}'\
        )
    print(f'  {"-"*14}  {"-"*7}  {"-"*7}  {"-"*8}  {"-"*7}  {"-"*9}')
    for col in ['total', 'phys', 'pf_ens', 'beam', 'selector']:
        _hw_df[col] = pd.to_numeric(_hw_df[col])
    means = _hw_df[['total','phys','pf_ens','beam','selector']].mean()
    print(
        f'  {"MEAN":<14}  {means["total"]:>6.2f}s'\
        f'  {means["phys"]:>6.2f}s'\
        f'  {means["pf_ens"]:>7.2f}s'\
        f'  {means["beam"]:>6.2f}s'\
        f'  {means["selector"]:>8.3f}s'\
    )
    print(f'{"-" * 72}')
    _slowest_h = _hw_df.loc[_hw_df['total'].idxmax(), 'wid']
    _fastest_h = _hw_df.loc[_hw_df['total'].idxmin(), 'wid']
    print(f'  Slowest well : {_slowest_h}  ({_hw_df["total"].max():.2f}s)')
    print(f'  Fastest well : {_fastest_h}  ({_hw_df["total"].min():.2f}s)')
    print(f'  Total wall time: {str(timedelta(seconds=int(_t_heuristic_total)))}  ({_t_heuristic_total:.1f}s)')
    print(f'{"-" * 72}\n')
    print(f'[TIMER] Heuristic inference: {_t_heuristic_total:.1f}s')

## 6.3 Blending

In [ ]:
_t_blend = time.perf_counter()

# Slightly more weight on physics/IMM path vs original (0.7 → 0.75)
# because IMM is more accurate than raw PF for smooth wells
sub = (
    sub_1.merge(sub_2, on='id', suffixes=('_1', '_2'))
         .assign(tvt=lambda x: 0.25 * x['tvt_1'] + 0.75 * x['tvt_2'])
         [['id', 'tvt']]
)
sub.to_csv('submission.csv', index=False)
_SECTION_TIMES['blending'] = time.perf_counter() - _t_blend
print(f'[TIMER] Blending: {_SECTION_TIMES["blending"]:.2f}s')
sub

# 7. Results

In [ ]:
fold_scores_df    = pd.DataFrame(fold_scores)
overall_scores_df = (
    pd.DataFrame({k: [v] for k, v in overall_scores.items()})
      .transpose()
      .sort_values(by=0, ascending=True)
)
order = overall_scores_df.index.tolist()

min_score  = overall_scores_df.values.flatten().min()
max_score  = overall_scores_df.values.flatten().max()
padding    = (max_score - min_score) * 0.5

fig, axs = plt.subplots(1, 2, figsize=(16, max(8, fold_scores_df.shape[1] * 0.65)))

boxplot = sns.boxplot(data=fold_scores_df, order=order, ax=axs[0], orient='h', color='grey')
axs[0].set_title('Fold RMSE')

barplot = sns.barplot(
    x=overall_scores_df.values.flatten(),
    y=overall_scores_df.index,
    ax=axs[1], color='grey'
)
axs[1].set_title('Overall RMSE')
axs[1].set_xlim(left=min_score - padding, right=max_score + padding)

for i, (score, model) in enumerate(zip(overall_scores_df.values.flatten(), overall_scores_df.index)):
    if 'stacking' in model.lower():
        color = 'lime'
    elif 'lasso' in model.lower() and 'elasticnet' not in model.lower():
        color = 'orange'
    elif 'elasticnet' in model.lower():
        color = 'deepskyblue'
    elif 'ridge' in model.lower():
        color = 'cyan'
    else:
        color = 'grey'
    barplot.patches[i].set_facecolor(color)
    boxplot.patches[i].set_facecolor(color)
    barplot.text(score, i, round(score, 3), va='center')

# ── Color legend ──────────────────────────────────────────────────────────────
# lime        = stacking (Ridge+Lasso+ElasticNet) ensemble
# deepskyblue = elasticnet  |  orange = lasso  |  cyan = ridge  |  grey = base
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lime',        label='Stacking (R+L+EN)'),
    Patch(facecolor='deepskyblue', label='ElasticNet'),
    Patch(facecolor='orange',      label='Lasso'),
    Patch(facecolor='cyan',        label='Ridge'),
    Patch(facecolor='grey',        label='Base model'),
]
axs[1].legend(handles=legend_elements, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

# ── Full Pipeline Timing Summary ──────────────────────────────────────────────
_pipeline_elapsed = time.perf_counter() - _PIPELINE_START
_SECTION_TIMES['TOTAL_PIPELINE'] = _pipeline_elapsed

_section_order = [
    'data_loading',
    'lightgbm_training',
    'catboost_training',
    'ensembling',
    'postprocessing',
    'ml_inference',
    'heuristic_inference',
    'blending',
    'TOTAL_PIPELINE',
]
_section_labels = {
    'data_loading':          'Data loading & feature eng.',
    'lightgbm_training':     'LightGBM training',
    'catboost_training':     'CatBoost training',
    'ensembling':            'Ensembling (Ridge+Lasso+EN)',
    'postprocessing':        'Post-processing',
    'ml_inference':          'ML inference (sub_1)',
    'heuristic_inference':   'Heuristic inference (sub_2)',
    'blending':              'Final blending',
    'TOTAL_PIPELINE':        'TOTAL PIPELINE',
}

print(f'\n{"=" * 58}')
print('FULL PIPELINE — SECTION TIMING REPORT')
print(f'{"=" * 58}')
print(f'  Started : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Elapsed : {str(timedelta(seconds=int(_pipeline_elapsed)))} ({_pipeline_elapsed:.1f}s)')
print(f'{"-" * 58}')
print(f'  {"Section":<32}  {"Time":>8}  {"% total":>8}')
print(f'  {"-"*32}  {"-"*8}  {"-"*8}')
for key in _section_order:
    if key not in _SECTION_TIMES:
        continue
    t   = _SECTION_TIMES[key]
    pct = 100.0 * t / max(_pipeline_elapsed, 1e-9)
    sep = '=' if key == 'TOTAL_PIPELINE' else ' '
    lbl = _section_labels.get(key, key)
    print(f'  {lbl:<32}  {t:>7.1f}s  {pct:>7.1f}%')
print(f'{"=" * 58}\n')

# ── Per-well timing heatmap (feature engineering, if available) ───────────────
if _WELL_TIMINGS:
    _wt_df = pd.DataFrame(_WELL_TIMINGS).T[['total','pf','kf','rbpf','imm','beam','ncc','spatial']].astype(float)
    _wt_df = _wt_df.sort_values('total', ascending=False)

    fig2, ax2 = plt.subplots(figsize=(12, max(4, len(_wt_df) * 0.35)))
    import seaborn as sns
    _norm = _wt_df.div(_wt_df.max())
    sns.heatmap(
        _norm, ax=ax2, cmap='YlOrRd', linewidths=0.4, linecolor='#ddd',
        annot=_wt_df.round(2), fmt='.2f', annot_kws={'size': 7},
        cbar_kws={'label': 'Relative time (0=fastest, 1=slowest)'}
    )
    ax2.set_title(
        f'Feature Engineering — Per-Well Stage Timing (seconds)\nSorted by total time  |  {len(_wt_df)} wells', fontsize=11, pad=10)
    ax2.set_xlabel('Pipeline stage')
    ax2.set_ylabel('Well ID')
    plt.tight_layout()
    plt.show()

# ── Heuristic inference timing chart ─────────────────────────────────────────
if '_hw_df' in dir() and len(_hw_df):
    _hw_plot = _hw_df.set_index('wid')[['pf_ens','beam','phys','selector']].sort_values('pf_ens', ascending=True)
    _hw_plot.plot(
        kind='barh', stacked=True, figsize=(10, max(4, len(_hw_plot) * 0.38)),\
        color=['#4e79a7','#f28e2b','#59a14f','#bab0ac'],\
        title=f'Heuristic Inference — Per-Well Stage Timing ({len(_hw_plot)} wells)\n'\
              f'(pf_ens = PF+KF+IMM ensemble;  beam = beam search)'\
    )
    plt.xlabel('Time (seconds)')
    plt.tight_layout()
    plt.show()